In [1]:
from loader import load_dataset, distance_from_class_index
import torch
from transformers import AutoImageProcessor, ConvNextV2ForImageClassification
from tqdm import tqdm
import torch.nn.functional as F

/home/austen/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_PATH = "/home/austen/GeoDataset/dataset_sharded"
BATCH_SIZE = 4
WORKERS = 2
S2_LEVEL = "L6"
LEARNING_RATE = 5e-5
PRETRAINED_MODEL_ID = "facebook/convnextv2-base-22k-224"
EPOCHS = 1
DIST_LOSS_WEIGHT = 0.30


In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f" - GPU {i}: {torch.cuda.get_device_name(i)}")

Device: cuda
GPU count: 2
 - GPU 0: NVIDIA GeForce RTX 3060
 - GPU 1: NVIDIA GeForce RTX 3060


In [4]:
dataset, class_centers, num_classes = load_dataset(DATASET_PATH, s2_level=S2_LEVEL, shuffle=False)
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    num_workers=WORKERS,
    pin_memory=True,
    prefetch_factor=2
)

print("Number of classes:", num_classes)

# Quick sanity check
for imgs, labels, coords in loader:
    print("Batch images:", imgs.shape)
    print("Batch labels:", labels.shape)
    print("Batch coords:", coords.shape)
    break

Number of classes: 3740
Batch images: torch.Size([4, 3, 768, 768])
Batch labels: torch.Size([4])
Batch coords: torch.Size([4, 2])


In [5]:
processor = AutoImageProcessor.from_pretrained(PRETRAINED_MODEL_ID, use_fast=True)

from transformers import AutoImageProcessor
processor = AutoImageProcessor.from_pretrained("facebook/convnext-tiny-224")

model = ConvNextV2ForImageClassification.from_pretrained(
    PRETRAINED_MODEL_ID,
    num_labels=num_classes,
    ignore_mismatched_sizes=True
).to(DEVICE)

# Enable DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler("cuda")

Some weights of ConvNextV2ForImageClassification were not initialized from the model checkpoint at facebook/convnextv2-base-22k-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([3740]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 1024]) in the checkpoint and torch.Size([3740, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using 2 GPUs


In [6]:
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    num_batches = 0

    class_centers = class_centers.to(DEVICE)

    for imgs, labels, coords in tqdm(loader, desc=f"Epoch {epoch + 1}"):
        imgs, labels, coords = imgs.to(DEVICE), labels.to(DEVICE), coords.to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(imgs)

        probs = torch.softmax(outputs.logits.float(), dim=1)
        dists = torch.cdist(coords.float(), class_centers.float())

        ce_loss = F.cross_entropy(outputs.logits, labels)
        dist_loss = (probs * dists).sum(dim=1).mean()

        loss = (1 - DIST_LOSS_WEIGHT) * ce_loss + DIST_LOSS_WEIGHT * dist_loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches if num_batches > 0 else 0
    print(f"Epoch {epoch + 1} | Average Loss: {avg_loss:.4f}")

Epoch 1: 103it [00:50,  2.05it/s]


KeyboardInterrupt: 